In [1]:
%load_ext autoreload
%autoreload 2

In [23]:
import os 
import json
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()  # reads variables from a .env file and sets them in os.environ

# Code of your application, which uses environment variables (e.g. from `os.environ` or
# `os.getenv`) as if they came from the actual environment.

from pyvis.network import Network
from math_assistant_agent.data import (
    fetch_math_dataset, 
    build_graph_records, 
    save_graph_json
)

In [3]:
from pprint import pprint

In [4]:
STACK_EXCHANGE_API_KEY = os.getenv("STACK_EXCHANGE_API_KEY", None)

In [5]:
raw_items = fetch_math_dataset(
    api_key=STACK_EXCHANGE_API_KEY, num_questions=100
)  # for larger pulls

Buscando 100 questões estruturadas do Math Stack Exchange...


In [6]:
graph_data = build_graph_records(raw_items)

In [21]:
print(""" 
print(type(graph_data))
print(graph_data.keys())
print(len(graph_data["nodes"]), len(graph_data["edges"]))
pprint(graph_data["nodes"][0])
pprint(graph_data["edges"][0])
      
""")

print(type(graph_data))
print(graph_data.keys())
print(len(graph_data["nodes"]), len(graph_data["edges"]))
pprint(graph_data["nodes"][0])
pprint(graph_data["edges"][0])

 
print(type(graph_data))
print(graph_data.keys())
print(len(graph_data["nodes"]), len(graph_data["edges"]))
pprint(graph_data["nodes"][0])
pprint(graph_data["edges"][0])


<class 'dict'>
dict_keys(['nodes', 'edges'])
339 430
{'id': 'question_182412',
 'label': 'Question',
 'properties': {'link': 'https://mathoverflow.net/questions/182412/why-do-roots-of-polynomials-tend-to-have-absolute-value-close-to-1',
                'question_id': 182412,
                'score': 446,
                'text': 'While playing around with Mathematica I noticed that '
                        'most polynomials with real coefficients seem to have '
                        'most complex zeroes very near the unit circle. For '
                        'instance, if we plot all the roots of a polynomial of '
                        'degree 300 with coefficients chosen randomly from the '
                        'interval $[27, 42]$, we get something like this:\n'
                        '\n'
               

In [29]:
GRAPH_PATH = "/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data"
INGESTION_DATE = datetime.now().strftime("%Y-%m-%d-%H")

In [30]:
save_graph_json(
    graph_data, 
    path=f"{GRAPH_PATH}/graph_math_kb_{INGESTION_DATE}.json"
)

✅ Grafo de conhecimento salvo em: /home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_kb_2026-07-17-16.json


'/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_kb_2026-07-17-16.json'

In [43]:
def render_graph(graph_data, output_path="knowledge_graph.html"):
    """
    Recebe um dicionário com 'nodes' e 'edges' e gera um arquivo HTML interativo.
    """
    print(f"Gerando visualização para {len(graph_data.get('nodes', []))} nós e {len(graph_data.get('edges', []))} arestas...")
    
    # Inicializa a rede do PyVis. 
    # directed=True é importante para vermos a direção da seta (ex: HAS_ACCEPTED_ANSWER)
    net = Network(
        height="800px", 
        width="100%", 
        bgcolor="#222222", 
        font_color="white", 
        directed=True,
        notebook=False # Mude para True se for renderizar direto na célula do Jupyter
    )

    # 1. Adicionando os Nós (Nodes)
    for node in graph_data.get("nodes", []):
        node_id = node["id"]
        label = node["label"]
        props = node.get("properties", {})
        
        # Formatando o texto que vai aparecer quando passar o mouse por cima (hover)
        # Vamos pegar o título e truncar o texto da pergunta para não poluir a tela
        title = props.get("title", "Sem Título")
        score = props.get("score", "N/A")
        views = props.get("view_count", "N/A")
        
        hover_text = (
            f"<b>Label:</b> {label}<br>"
            f"<b>Title:</b> {title}<br>"
            f"<b>Score:</b> {score} | <b>Views:</b> {views}"
        )
        
        # Diferenciando cores baseado no Label do nó
        cor = "#97C2FC" # Azul padrão
        if label == "Question":
            cor = "#FF9999" # Vermelho claro
        elif label == "Answer":
            cor = "#99FF99" # Verde claro
            
        # O rótulo visível no balão será o tipo + os primeiros caracteres do ID
        display_label = f"{label}\n({node_id.split('_')[-1]})"
        
        net.add_node(
            n_id=node_id, 
            label=display_label, 
            title=hover_text, 
            color=cor
        )

    # 2. Adicionando as Arestas (Edges)
    for edge in graph_data.get("edges", []):
        source = edge["source"]
        target = edge["target"]
        edge_type = edge["type"]
        
        net.add_edge(
            source=source, 
            to=target, 
            title=edge_type, # Texto ao passar o mouse na linha
            label=edge_type, # Texto visível na linha
            color="#888888"
        )

    # 3. Adicionando física para o grafo se organizar sozinho na tela
    net.set_options("""
    var options = {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -50,
          "centralGravity": 0.01,
          "springLength": 100,
          "springConstant": 0.08
        },
        "maxVelocity": 50,
        "solver": "forceAtlas2Based",
        "timestep": 0.35,
        "stabilization": {"iterations": 150}
      }
    }
    """)

    # 4. Salvando o arquivo HTML
    net.write_html(output_path)
    print(f"✅ Grafo renderizado com sucesso! Abra o arquivo '{output_path}' no seu navegador.")

def get_node_by_id(graph_data, node_id):
    """Busca e retorna o dicionário completo de um nó pelo seu ID."""
    for node in graph_data["nodes"]:
        if node["id"] == node_id:
            return node
    return None

def get_accepted_answer(graph_data, question_id):
    """Busca o texto da answer aceita conectada a uma pergunta."""
    answer_id = None
    
    # 1. Procurar a aresta (edge) que conecta a pergunta à answer
    for edge in graph_data["edges"]:
        if edge["source"] == question_id and edge["type"] == "HAS_ACCEPTED_ANSWER":
            answer_id = edge["target"]
            break # Paramos a busca assim que encontrar
            
    # 2. Se achou o ID da answer, buscar os dados completos do nó
    if answer_id:
        return get_node_by_id(graph_data, answer_id)
    
    return None

def get_most_popular_questions(graph_data, min_score=100):
    """Retorna uma lista com todos os nós de perguntas acima de um certo score."""
    populares = [
        node for node in graph_data["nodes"] 
        if node["label"] == "Question" and node["properties"].get("score", 0) > min_score
    ]
    return populares

In [33]:
render_graph(
    graph_data, 
    output_path=f"{GRAPH_PATH}/graph_math_kb_{INGESTION_DATE}.html"
)

Gerando visualização para 339 nós e 430 arestas...
✅ Grafo renderizado com sucesso! Abra o arquivo '/home/cdrleo/Desktop/ai-projects/math-assistant-agent/data/graph_math_kb_2026-07-17-16.html' no seu navegador.


In [44]:
############################################################################
# Testando a busca:
pergunta = get_node_by_id(graph_data, "question_109")

if pergunta:
    pprint(f"Título: {pergunta['properties']['title']}")
    pprint(f"Score: {pergunta['properties']['score']}")

print("\n###############################################\n")
# Testando a busca:
answer = get_accepted_answer(graph_data, "question_109")

if answer:
    # Mostra um pedaço do texto da answer
    pprint(f"ID da answer: {answer['id']}")
    pprint(f"Conteúdo: {answer['properties'].get('text', '')[:200]}...")

print("\n###############################################\n")

# Testando a busca:
top_perguntas = get_most_popular_questions(graph_data, min_score=200)
pprint(f"Encontrei {len(top_perguntas)} perguntas super populares!")
# pprint(top_perguntas)

'Título: What do epimorphisms of (commutative) rings look like?'
'Score: 126'

###############################################

'ID da answer: answer_161'
('Conteúdo: No, not every epimorphism of rings is a composition of '
 'localizations and surjections.\n'
 '\n'
 'An epimorphism of commutative rings is the same thing as a monomorphism of '
 'affine schemes. Monomorphisms are not ...')

###############################################

'Encontrei 12 perguntas super populares!'
